# DDPM Example Training Notebook

This script walks through the full workflow of training a model

In [ ]:
import torch
from ddpm import NoiseScheduler, UNet, train, find_lr, generate_image, noisy_image
from ddpm.dataset import load_mnist, get_noisy_loaders, NoisyMNIST,NoisyDataset
from ddpm.utils import load_unet, channel_list, model_name, path_name
from ddpm.viz import plot_generated

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Noise scheduler and data

if you want to train on a subset of the data, use get_noisy_loaders_filtered

In [ ]:
scheduler = NoiseScheduler(T=1000, beta_start=1e-4, beta_end=0.02)

train_set, test_set = load_mnist()
train_loader, test_loader = get_noisy_loaders(train_set, test_set, scheduler, batch_size=32)

train_set_noisy = NoisyMNIST(train_set, scheduler)
test_set_noisy = NoisyMNIST(test_set,  scheduler)

## 2. Build a UNet

`channels` sets the feature map depth at each encoder level.
The decoder mirrors this automatically.
`convs_per_level` is how many conv layers per resolution block.

### TO SET

In [ ]:
channel0 = 64
cpl = 2 

# model_path = '/home/scur0036/diffusion-models-project/models/base_C0_128_convs_2wd_1e-6_long.pkl'

In [ ]:
channels = channel_list(channel0) 
                      # convs per level

unet = UNet(channels=channels, convs_per_level=cpl).to(device)
# if you want to train an existing model already, use load_unet
# unet = load_unet(model_path, channels=channels, convs_per_level=cpl)
# where model_path is the path to the state_dict .pkl file

print(f"Model: {model_name(channel0, cpl)}")
print(f"Parameters: {sum(p.numel() for p in unet.parameters()):,}")

## 3. Find a learning rate

if we send a job with train_on_cluster and don't set a learning rate, the lr is automatically determined by this scheme. Is more effective bc running the following cell on a cpu is not feasible often

In [ ]:
# suggested_lr = find_lr(unet, train_loader)
# lr = suggested_lr * 0.5
# print(f"Suggested LR: {suggested_lr:.2e} → using {lr:.2e}")

## 4. Train

### TO SET

In [ ]:
# if we train in the notebook
# save_path = path_name(channel0, cpl,add_desc="wd_1e-6")   # e.g. "base_C0_64_convs_2.pkl"

job_name = 'unet_C0_128_convs_2'

either we train in this notebook itself

In [ ]:


# train_losses, test_losses = train(
#     unet, train_loader, test_loader,
#     epochs=50,
#     lr=lr,
#     weight_decay=1e-6,
#     early_stopping_patience=10,
#     save_path=save_path,
# )

### Send training to cluster
alternatively, a separate job can be send to a node to train the model

In [ ]:
import cluster
from cluster import train_on_cluster, job_status,train_and_generate_on_cluster,generate_on_cluster
# if you didn't set up your environment with setup.py, look at the notebook example_usage_ddpm chapter 0 on how to set the global variables of cluster to make it work for your project_dir and your environment

# submit

job_id, save_path = train_on_cluster(
    unet,train_set_noisy,test_set_noisy,
    epochs=50,
    job_name=job_name,
)


or directly generate images after the training is finished in the same job

In [ ]:
# job_id = train_and_generate_on_cluster(
#     model=unet,
#     train_dataset=train_set,
#     test_dataset=test_set,
#     epochs=100,
#     n_images=8,
#     stochasticity=1.0,
#     job_name=job_name,   # determines the folder under models/
#     time='02:00:00',
# )

### check job status

In [ ]:
job_status(job_id)

or simply

In [ ]:
! squeue

load trained model

In [ ]:
from cluster import get_save_path,get_losses_path
# load weights
unet.load_state_dict(torch.load(get_save_path(job_name),map_location=device))

# load losses
losses = torch.load(get_losses_path(job_name))
train_losses,test_losses = losses['train'],losses['test']


Plot the loss curves manually (training doesn't do this anymore):

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(train_losses,test_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses,  label='Test')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Loss curve')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss(train_losses,test_losses)

## 6. Generate images

this can either be done locally

In [ ]:
# x = generate_image(unet, scheduler, stochasticity=1.0, n_images=8)
# plot_generated(x, ncol=4)

or also be send as a job to the cluster. the model is taken from the job_name repository, which then includes a trained.pkl.

maybe not the most elegant structure, but that can also still be changed i guess.

In [ ]:

job_id = generate_on_cluster(
    job_name=job_name,
    n_images=16,
    stochasticity=0.8,
)

# once done


In [ ]:
job_status(job_id)

In [ ]:
x = torch.load(cluster.get_generated_path(job_name))
plot_generated(x, ncol=4)

### Intermediates

In [ ]:
x_final, intermediates = generate_image(
    unet, scheduler, stochasticity=1.0, n_images=1, return_intermediates=True
)

# Plot every 100th step
fig, axes = plt.subplots(1, 11, figsize=(22, 2))
steps_to_show = list(range(0, 1000, 100)) + [999]
for ax, idx in zip(axes, steps_to_show):
    img = intermediates[idx].squeeze()
    if img.min() < 0:
        img = (img + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.set_title(f't={1000 - idx}', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()